# Canonical (ATG) vs. non-canonical (non-ATG) start codon detection

Two complementary assessments of how well each gene caller recovers canonical (ATG) versus
non-canonical (non-ATG) start codons, plus the ground-truth counts they rest on.

**A. Border-excluded (all three models).** A start codon sitting at the read's very first
in-frame codon is ambiguous from coordinates alone: a CDS fragment beginning there could mean
"a gene starts at this position" or "coding continues from upstream of this read". FGS emits no
explicit start-codon call, so for FGS the two are indistinguishable. Assessment A therefore drops
those positions from **both** the ground truth and the predictions, leaving a like-for-like
comparison that FGS can take part in.

**B. Border-inclusive (DeepCDS and Prodigal only).** Both of these models *do* say explicitly
where they think a start codon is - DeepCDS through the per-codon label 2, surfaced in its GFF as
`start=start_codon` (vs `internal_region`); Prodigal through `start_type=ATG|GTG|TTG` (vs `Edge`,
which means it declined to call a start because the gene runs off the sequence boundary). With
that signal there is no ambiguity left to avoid, so assessment B applies no positional filtering
at all: ground truth is every genuine start codon in the read including those at the very first
codon, and a predicted start is exactly a position where the model claims one.

Both assessments report:

1. **Pooled, unweighted performance.** TP/FP/FN summed over every read of every TT11 test genome,
   with precision/recall/F1 computed once from those totals - a micro-average where each start
   codon occurrence counts equally and no genome is weighted or normalised.
2. **A genome-level cluster bootstrap standard error**, matching
   `cds_level_metagenome_model_comparison.ipynb` (`pooled_scores`/`bootstrap_pooled_std`,
   `N_BOOTSTRAP = 1000`, `np.random.default_rng(42)`): resample whole genomes with replacement,
   sum each replicate's raw counts into one pooled count, recompute the pooled score. Reads from
   one genome are correlated, so genomes - not reads - are the resampling unit.
3. **The number of ATG vs. non-ATG occurrences** the scores are based on.

**The 60bp dataset is excluded.** Its standard (CDS > 60bp) testset and prediction files are
empty, and the `_30` (CDS >= 30bp) fallback exists only for DeepCDS, so it admits no like-for-like
comparison.

**Data requirements.** Assessment A reads the post-processed prediction pickles; assessment B
reads the **raw** prediction GFFs, because `postprocess_model_predictions.py` and
`postprocess_prodigal_preds.py` both discard the start-codon attribute when building those
pickles. Read sequences and per-codon `rf*_labels` come from
`reads_processed/test/{data_type}/csv/{accession}.csv.gz`. **Run this where those directories are
populated (e.g. on the cluster).**

In [89]:
import ast
import os
import pickle
import sys
from collections import Counter

import numpy as np
import pandas as pd
from tqdm import tqdm

sys.path.insert(0, '..')
from plot_config import MODEL_DISPLAY_NAMES

In [90]:
#project_root = "../../../.."
project_root = "/tmp/nrt204/FragmentPredictor"
all_test_accessions = open(f"{project_root}/data/processed_data/genome_partitions/test_partition_accessions.txt").read().splitlines()
all_test_accessions = all_test_accessions[:10]

# Toggle: set True to load pre-computed results from disk (fast), False to run full data processing
LOAD_PROCESSED_DATA = True

# Load genome info for TT classification
genome_info_df = pd.read_csv(f"{project_root}/data/processed_data/dataset_information/genomes_info_with_partitions.csv")
genome_info_df = genome_info_df.set_index('accession')

TT4_FAMILIES = ['Mycoplasmataceae']

all_genomes = {}
for acc in all_test_accessions:
    if acc in genome_info_df.index:
        row = genome_info_df.loc[acc]
        translation_table = 4 if row['family'] in TT4_FAMILIES else 11
        all_genomes[acc] = {
            'family': row['family'],
            'gc_content': row['gc_content'],
            'translation_table': translation_table
        }

print(f"Total test accessions: {len(all_test_accessions)}")

# 60bp is deliberately NOT included - see the notebook header.
read_lengths = [75, 100, 150, 300, 700, 1000]

model_names = ["fgs_complete", "prodigal", "deep_cds"]

# Assessment B needs an explicit, per-position start-codon call from the model itself. FGS emits
# none, so it takes part in assessment A only. DeepCDS exposes its per-codon label 2 as
# `start=start_codon` in its raw GFF; Prodigal exposes `start_type=ATG|GTG|TTG` (or `Edge` when it
# declines to call a start at a sequence boundary).
border_inclusive_models = ["prodigal", "deep_cds"]

# The DeepCDS run whose raw GFFs assessment B parses - must match the run used for the
# post-processed pickles in load_model_preds, or the two assessments would score different models.
DEEPCDS_RUN_NAME = "full_model_all_genomes_seed_42_trained_final_8M_no_dropout"

# Minimum CDS fragment length. postprocess_model_predictions.py / postprocess_prodigal_preds.py
# keep only fragments strictly longer than this when building the pickles assessment A reads, so
# assessment B applies the identical filter while parsing the raw GFFs - otherwise B would be
# scored against a larger prediction set than A and the two would not be comparable.
MIN_CDS_LENGTH = 60

codon_types = ["ATG", "non_ATG"]
codon_type_labels = {"ATG": "Canonical (ATG)", "non_ATG": "Non-canonical (non-ATG)"}
metric_keys = ('precision', 'recall', 'f1')
metric_display = {'f1': 'F1', 'precision': 'Precision', 'recall': 'Sensitivity'}

model_display_names = MODEL_DISPLAY_NAMES

_script_name = "canonical_vs_non_canonical_start_codon_test"
_cache_dir = f"{project_root}/data/processed_data/report_results/without_errors/{_script_name}"

Total test accessions: 10


## Loaders

`load_testset` / `load_model_preds` / `load_read_sequences` / `load_read_labels` serve assessment A
and are unchanged from the sibling start-codon notebooks. The two `load_*_claimed_starts` loaders
are new, and serve assessment B: they read each model's **raw** GFF, because both postprocessing
scripts drop the start-codon attribute when writing the pickles
(`postprocess_model_predictions.py` parses the attribute column but keeps only `group_id`;
`postprocess_prodigal_preds.py` has its `attr_desc` line commented out).

In [91]:
def load_testset(test_accession, data_type, project_root=project_root):
    """Ground-truth CDS fragments (>60bp) and the read-name list for one test genome."""
    with open(f"{project_root}/data/processed_data/testset_processed/{data_type}/{test_accession}/testset_dict.pkl", "rb") as f:
        testset_dict = pickle.load(f)
    with open(f"{project_root}/data/processed_data/testset_processed/{data_type}/{test_accession}/read_names_list.pkl", "rb") as f:
        read_names_list = pickle.load(f)
    return read_names_list, testset_dict


def load_model_preds(test_accession, data_type, model_name, project_root=project_root):
    """Post-processed predictions (>60bp CDS fragments) for one model on one test genome."""
    pred_paths = {
        "prodigal": f"{project_root}/data/processed_data/predictions/processed_predictions/prodigal_preds/{data_type}/{test_accession}/model_preds_dict.pkl",
        "fgs_complete": f"{project_root}/data/processed_data/predictions/processed_predictions/fgs_preds/{data_type}/{test_accession}.pkl",
        "deep_cds": f"{project_root}/data/processed_data/predictions/processed_predictions/DeepCDS/model_without_errors/{data_type}/{DEEPCDS_RUN_NAME}/{test_accession}/model_preds_dict.pkl",
    }
    with open(pred_paths[model_name], "rb") as f:
        preds = pickle.load(f)
    return preds


def load_read_sequences(test_accession, data_type, project_root=project_root):
    """Map read_name -> raw read sequence, needed to look up the nucleotides at a start codon."""
    reads_df = pd.read_csv(
        f"{project_root}/data/processed_data/reads_processed/test/{data_type}/csv/{test_accession}.csv.gz",
        compression="gzip",
        usecols=["read_name", "read"],
    )
    return dict(zip(reads_df["read_name"], reads_df["read"]))


def load_read_labels(test_accession, data_type, project_root=project_root):
    """Map read_name -> {0: rf0_labels, 1: rf1_labels, 2: rf2_labels} (parsed per-codon label
    lists: 0=non-coding, 1=coding interior, 2=start, 3=stop). Used to verify whether a
    ground-truth CDS fragment's recorded start position is a genuinely observed start codon
    rather than a read-boundary artifact - see is_internal_start."""
    path = f"{project_root}/data/processed_data/reads_processed/test/{data_type}/csv/{test_accession}.csv.gz"
    reads_df = pd.read_csv(path, compression="gzip", usecols=["read_name", "rf0_labels", "rf1_labels", "rf2_labels"])
    read_labels = {}
    for read_name, l0, l1, l2 in zip(reads_df["read_name"], reads_df["rf0_labels"], reads_df["rf1_labels"], reads_df["rf2_labels"]):
        read_labels[read_name] = {0: ast.literal_eval(l0), 1: ast.literal_eval(l1), 2: ast.literal_eval(l2)}
    return read_labels


def _parse_gff_attributes(attr_field):
    """GFF column 9 ('a=1;b=2;') -> dict. Values may themselves contain '=' (DeepCDS writes
    ref=[[...]]), so only the first '=' separates key from value."""
    attrs = {}
    for item in attr_field.rstrip().rstrip(";").split(";"):
        if "=" in item:
            key, value = item.split("=", 1)
            attrs[key] = value
    return attrs


def load_prodigal_claimed_starts(test_accession, data_type, project_root=project_root,
                                 min_cds_length=MIN_CDS_LENGTH):
    """Positions where Prodigal EXPLICITLY claims a start codon, parsed from its raw GFF.

    Returns {read_name: {(start_position, rf), ...}}.

    Prodigal's GFF attribute column carries `start_type=ATG|GTG|TTG|Edge`. `Edge` means the gene
    runs off the sequence boundary and Prodigal declined to call a start (paired with
    `partial=1x`), so those are NOT claims and are skipped. Verified empirically that a 3'-partial
    gene (`partial=01`) still carries a real start_type, so `start_type != Edge` - not `partial` -
    is the correct test for "a start is claimed here".

    Reading frame is derived from the start position the same way postprocess_prodigal_preds.py
    does it, NOT from GFF column 8: Prodigal writes phase 0 on every CDS line (phase is relative to
    the CDS itself), which carries no read-relative frame information.

    Comment lines are skipped explicitly rather than by discarding the first line. The pipeline's
    `grep '\t+\t'` step (predict_with_prodigal.py) already strips every comment line from these
    files, so postprocess_prodigal_preds.py's `file.readline()` actually discards the first real
    CDS record of each genome; this loader does not inherit that.
    """
    path = (f"{project_root}/data/processed_data/predictions/raw_predictions/prodigal_preds/"
            f"{data_type}/{test_accession}/{test_accession}.gff")
    claimed = {}
    with open(path) as f:
        for line in f:
            if line.startswith("#"):
                continue
            fields = line.rstrip("\n").split("\t")
            if len(fields) < 9 or fields[2] != "CDS" or fields[6] != "+":
                continue
            cds_start, cds_end = int(fields[3]), int(fields[4])
            if cds_end - cds_start + 1 <= min_cds_length:
                continue
            if _parse_gff_attributes(fields[8]).get("start_type", "Edge") == "Edge":
                continue
            read_name = fields[0].split("|")[0]
            rf = {1: 0, 2: 1, 0: 2}[cds_start % 3]
            claimed.setdefault(read_name, set()).add((cds_start, str(rf)))
    return claimed


def load_deepcds_claimed_starts(test_accession, data_type, project_root=project_root,
                                model_run_name=DEEPCDS_RUN_NAME, min_cds_length=MIN_CDS_LENGTH):
    """Positions where DeepCDS EXPLICITLY claims a start codon, parsed from its raw GFF.

    Returns {read_name: {(start_position, rf), ...}}.

    write_enhanced_gff (predict_with_DeepCDS.py) writes `start=<start_type>` on every CDS line,
    where start_type is 'start_codon' (the model predicted label 2 at that position),
    'internal_region' (coding began without a predicted start) or 'indel_start'. Only
    'start_codon' is a start claim. 'indel_start' is not counted: it marks a fragment resuming
    after a predicted indel, not a start codon, and without_errors data produces none anyway.

    The standalone `start_codon` feature lines in the same GFF are deliberately not used - they
    are written with '.' in the frame column, so they cannot be matched to a reading frame.
    Reading frame here comes from GFF column 8, which write_enhanced_gff fills with segment.frame.
    """
    path = (f"{project_root}/data/processed_data/predictions/raw_predictions/DeepCDS/"
            f"model_without_errors/{data_type}/{model_run_name}/predictions_{test_accession}.gff")
    claimed = {}
    with open(path) as f:
        for line in f:
            if line.startswith("#"):
                continue
            fields = line.rstrip("\n").split("\t")
            if len(fields) < 9 or fields[2] != "CDS":
                continue
            cds_start, cds_end = int(fields[3]), int(fields[4])
            if cds_end - cds_start + 1 <= min_cds_length:
                continue
            if _parse_gff_attributes(fields[8]).get("start") != "start_codon":
                continue
            claimed.setdefault(fields[0], set()).add((cds_start, str(int(fields[7]))))
    return claimed


CLAIMED_START_LOADERS = {
    "prodigal": load_prodigal_claimed_starts,
    "deep_cds": load_deepcds_claimed_starts,
}

In [92]:
def is_internal_start(start, rf, labels_by_rf=None):
    '''Return True if a CDS fragment's recorded start position should NOT be treated as an
    observed start codon.

    A fragment whose start is not at the read's very first in-frame codon (start != rf+1) can only
    arise from a genuine label transition in how cds_coords is built (verified empirically: every
    such fragment's recorded start corresponds to an actual label==2 position), so it is
    unconditionally treated as a real start.

    At the read's very first in-frame codon (start == rf+1) the codon itself is fully present in
    the read, but its meaning is ambiguous from coordinates alone: either the gene genuinely starts
    there, or it started upstream of the read and this is just where the fragment got clipped.
      - labels_by_rf given (assessment B's ground truth): resolved definitively by checking whether
        the per-codon ground-truth label at that position really is 2.
      - labels_by_rf omitted (assessment A, both sides): treated as not-a-start. Assessment A drops
        these positions from ground truth AND predictions alike, which is what makes it a
        like-for-like comparison FGS can join - FGS emits no start-codon call, so for FGS the
        ambiguity is unresolvable. Excluding them symmetrically matters: admitting them into the
        ground truth while the prediction side cannot express them would make every one an
        unwinnable false negative, even for a perfect predictor.

    For disrupted_rf entries (without_errors data never produces these) there is no single reading
    frame's label array to check, so the boundary-position heuristic is kept unconditionally.'''
    if rf == 'disrupted_rf':
        return start in {1, 2, 3}
    rf_int = int(rf)
    if start != rf_int + 1:
        return False
    if labels_by_rf is None:
        return True
    labels = labels_by_rf.get(rf_int)
    if labels is None:
        return True
    codon_idx = (start - rf_int - 1) // 3
    if codon_idx < 0 or codon_idx >= len(labels):
        return True
    return labels[codon_idx] != 2


def get_real_starts(cds_coords, labels_by_rf=None):
    '''Set of (start, rf) for the CDS starts that count as observed start codons. Pass
    ground-truth labels_by_rf to admit label-verified starts at the read's first codon
    (assessment B); omit it to exclude every such position (assessment A). See is_internal_start.'''
    return {(start, rf) for start, stop, rf in cds_coords if not is_internal_start(start, rf, labels_by_rf)}


def is_border_start(start, rf):
    '''True if this start sits at the read's very first in-frame codon - the position assessment A
    excludes and assessment B keeps.'''
    if rf == 'disrupted_rf':
        return start in {1, 2, 3}
    return start == int(rf) + 1


def get_start_codon(read, start):
    '''3-nt codon at a 1-based start position within a read.'''
    return read[start - 1: start + 2]


def split_by_codon_type(starts, read):
    '''Split a set of (start, rf) into (atg_starts, non_atg_starts) using the actual read sequence.'''
    atg, non_atg = set(), set()
    for start, rf in starts:
        target = atg if get_start_codon(read, start) == "ATG" else non_atg
        target.add((start, rf))
    return atg, non_atg


def start_counts_by_codon_type(pred_starts, actual_starts, read):
    '''TP/FP/FN for start codon detection (exact position match), split by ATG vs non-ATG.

    Both sides are classified by the codon actually sitting at that position in the read (ground
    truth, never model output), so a correct match is always classified consistently between
    prediction and truth. FN counts a true start as missed if no prediction of ANY codon type lands
    on it; FP counts a prediction as a miss of its own type if it lands on no real start of any
    type. Shared by both assessments - they differ only in how the two start sets are built.'''
    pred_atg, pred_non_atg = split_by_codon_type(pred_starts, read)
    actual_atg, actual_non_atg = split_by_codon_type(actual_starts, read)

    counts = {}
    for label, actual_set, pred_set in [("ATG", actual_atg, pred_atg), ("non_ATG", actual_non_atg, pred_non_atg)]:
        counts[label] = (len(pred_set & actual_set),          # tp
                         len(pred_set - actual_starts),       # fp
                         len(actual_set - pred_starts))       # fn
    return counts

## Main loop

Per read length and genome, both assessments are accumulated in the same pass over the reads:

- **A (border-excluded, all 3 models):** ground-truth starts and predicted starts both come from
  `cds_coords` with `get_real_starts(..., labels_by_rf=None)`, so the read's first in-frame codon
  is dropped from both sides.
- **B (border-inclusive, DeepCDS + Prodigal):** ground-truth starts come from
  `get_real_starts(..., labels_by_rf=<this read's labels>)`, which additionally admits a start at
  the first codon when the ground-truth label there really is 2; predicted starts come from the
  model's own explicit start-codon claims in its raw GFF.

For each, three things are stored: pooled TP/FP/FN (`results_*`), the same counts split per genome
(`per_genome_counts_*`, the bootstrap's resampling population - every processed genome kept
unconditionally, since a genome with no true start of a codon type still contributes false
positives of that type to the pooled precision), and the model-independent ground-truth occurrence
counts (`actual_start_counts_*`).

`border_start_counts` records the extra stratum B admits and A does not, so its size is visible
rather than implicit. TT4 genomes (Mycoplasmataceae) are excluded throughout.

In [93]:
def _new_counts():
    return {'tp': 0, 'fp': 0, 'fn': 0}


if not LOAD_PROCESSED_DATA:
    # --- assessment A: border-excluded, all three models ---
    results_excl = {length: {m: {ct: _new_counts() for ct in codon_types} for m in model_names}
                    for length in read_lengths}
    per_genome_counts_excl = {length: {m: {ct: {} for ct in codon_types} for m in model_names}
                              for length in read_lengths}
    has_data_excl = {length: {m: False for m in model_names} for length in read_lengths}
    actual_start_counts_excl = {length: {ct: 0 for ct in codon_types} for length in read_lengths}

    # --- assessment B: border-inclusive, models with an explicit start-codon call ---
    results_incl = {length: {m: {ct: _new_counts() for ct in codon_types} for m in border_inclusive_models}
                    for length in read_lengths}
    per_genome_counts_incl = {length: {m: {ct: {} for ct in codon_types} for m in border_inclusive_models}
                              for length in read_lengths}
    has_data_incl = {length: {m: False for m in border_inclusive_models} for length in read_lengths}
    actual_start_counts_incl = {length: {ct: 0 for ct in codon_types} for length in read_lengths}

    # Diagnostics: the border-only stratum (true starts B admits and A drops), and how each model
    # fares on exactly that stratum - this is where Prodigal's refusal to call a start at a
    # sequence edge shows up explicitly rather than being absorbed into the combined figure.
    border_start_counts = {length: {ct: 0 for ct in codon_types} for length in read_lengths}
    border_only_counts = {length: {m: {ct: _new_counts() for ct in codon_types}
                                   for m in border_inclusive_models} for length in read_lengths}
    non_atg_codon_identity_counts = {length: Counter() for length in read_lengths}

    for length in read_lengths:
        data_type = f"without_errors_{length}bp"
        print(f"\nProcessing {data_type}...")

        for test_accession in tqdm(all_test_accessions, desc=f"{length}bp"):
            if all_genomes.get(test_accession, {}).get('translation_table', None) == 4:
                continue

            try:
                read_names_list, testset_dict = load_testset(test_accession, data_type)
                read_seqs = load_read_sequences(test_accession, data_type)
                read_labels = load_read_labels(test_accession, data_type)
            except Exception as e:
                print(f"Error loading testset/reads for {test_accession} at {length}bp: {e}")
                continue

            # Ground-truth start sets per read, built once and reused by every model below.
            truth_excl, truth_incl = {}, {}
            for read_name in read_names_list:
                actual = testset_dict.get(read_name, {}).get('cds_coords', [])
                read = read_seqs.get(read_name)
                if read is None or not actual:
                    continue
                labels_by_rf = read_labels.get(read_name, {})
                s_excl = get_real_starts(actual)
                s_incl = get_real_starts(actual, labels_by_rf)
                truth_excl[read_name] = s_excl
                truth_incl[read_name] = s_incl

                for store, starts in ((actual_start_counts_excl, s_excl), (actual_start_counts_incl, s_incl)):
                    atg, non_atg = split_by_codon_type(starts, read)
                    store[length]['ATG'] += len(atg)
                    store[length]['non_ATG'] += len(non_atg)

                border_atg, border_non_atg = split_by_codon_type(s_incl - s_excl, read)
                border_start_counts[length]['ATG'] += len(border_atg)
                border_start_counts[length]['non_ATG'] += len(border_non_atg)
                for start, rf in split_by_codon_type(s_incl, read)[1]:
                    non_atg_codon_identity_counts[length][get_start_codon(read, start)] += 1

            # ---------------- assessment A ----------------
            for model in model_names:
                try:
                    preds = load_model_preds(test_accession, data_type, model)
                except Exception as e:
                    print(f"Error loading {model} for {test_accession} at {length}bp: {e}")
                    continue
                has_data_excl[length][model] = True

                pg = {ct: _new_counts() for ct in codon_types}
                for read_name in read_names_list:
                    read = read_seqs.get(read_name)
                    if read is None:
                        continue
                    pred_starts = get_real_starts(preds.get(read_name, {}).get('cds_coords', []))
                    actual_starts = truth_excl.get(read_name, set())
                    if not pred_starts and not actual_starts:
                        continue
                    for ct, (tp, fp, fn) in start_counts_by_codon_type(pred_starts, actual_starts, read).items():
                        for key, val in (('tp', tp), ('fp', fp), ('fn', fn)):
                            results_excl[length][model][ct][key] += val
                            pg[ct][key] += val
                for ct in codon_types:
                    per_genome_counts_excl[length][model][ct][test_accession] = dict(pg[ct])

            # ---------------- assessment B ----------------
            for model in border_inclusive_models:
                try:
                    claimed = CLAIMED_START_LOADERS[model](test_accession, data_type)
                except Exception as e:
                    print(f"Error loading raw {model} GFF for {test_accession} at {length}bp: {e}")
                    continue
                has_data_incl[length][model] = True

                pg = {ct: _new_counts() for ct in codon_types}
                pg_border = {ct: _new_counts() for ct in codon_types}
                for read_name in read_names_list:
                    read = read_seqs.get(read_name)
                    if read is None:
                        continue
                    pred_starts = claimed.get(read_name, set())
                    actual_starts = truth_incl.get(read_name, set())
                    if not pred_starts and not actual_starts:
                        continue
                    for ct, (tp, fp, fn) in start_counts_by_codon_type(pred_starts, actual_starts, read).items():
                        for key, val in (('tp', tp), ('fp', fp), ('fn', fn)):
                            results_incl[length][model][ct][key] += val
                            pg[ct][key] += val

                    # Border-only stratum: restrict BOTH sides to first-codon positions, so this is
                    # a self-contained scoring of just the positions assessment A drops.
                    b_pred = {s for s in pred_starts if is_border_start(*s)}
                    b_actual = {s for s in actual_starts if is_border_start(*s)}
                    if b_pred or b_actual:
                        for ct, (tp, fp, fn) in start_counts_by_codon_type(b_pred, b_actual, read).items():
                            for key, val in (('tp', tp), ('fp', fp), ('fn', fn)):
                                pg_border[ct][key] += val
                for ct in codon_types:
                    per_genome_counts_incl[length][model][ct][test_accession] = dict(pg[ct])
                    for key in ('tp', 'fp', 'fn'):
                        border_only_counts[length][model][ct][key] += pg_border[ct][key]

else:
    print(f"Loading pre-computed results from:\n  {_cache_dir}")
    for _name in ["results_excl", "per_genome_counts_excl", "has_data_excl", "actual_start_counts_excl",
                  "results_incl", "per_genome_counts_incl", "has_data_incl", "actual_start_counts_incl",
                  "border_start_counts", "border_only_counts", "non_atg_codon_identity_counts"]:
        with open(os.path.join(_cache_dir, f"{_name}.pkl"), "rb") as f:
            globals()[_name] = pickle.load(f)

Loading pre-computed results from:
  /tmp/nrt204/FragmentPredictor/data/processed_data/report_results/without_errors/canonical_vs_non_canonical_start_codon_test


In [94]:
if not LOAD_PROCESSED_DATA:
    os.makedirs(_cache_dir, exist_ok=True)
    for _name in ["results_excl", "per_genome_counts_excl", "has_data_excl", "actual_start_counts_excl",
                  "results_incl", "per_genome_counts_incl", "has_data_incl", "actual_start_counts_incl",
                  "border_start_counts", "border_only_counts", "non_atg_codon_identity_counts"]:
        with open(os.path.join(_cache_dir, f"{_name}.pkl"), "wb") as f:
            pickle.dump(globals()[_name], f)
    print(f"Results saved to {_cache_dir}")

## Ground-truth occurrence counts

How many ATG vs. non-ATG start codons each assessment is actually scored against, their combined
total per read length (the whole annotated start codon count for that testset, since ATG and
non-ATG partition the ground truth), and how much of the difference between the two assessments
is the border stratum. Counts are **per read occurrence**, not per distinct gene:
reads are placed at 1x coverage, so one real start codon may be captured by zero, one or several
reads and each occurrence is counted separately. That is the right denominator here (a caller has
to find the start within each individual read it is handed), but it means the counts scale with
the number of simulated reads, which is why they shrink as read length grows.

**What the denominator is, and what it is not.** These counts come from the CDS fragment start
positions in `testset_dict.pkl`, not from a direct tally of start-codon labels in the reads. That
file keeps only fragments longer than 60bp (the criterion the model postprocessing applies to
predictions), so the counts are *start codons belonging to a CDS fragment longer than 60bp*, not
every start codon present in the test reads. The gap is large at short read lengths: on one genome,
723 of 4112 real start-codon labels are counted at 75bp (82% excluded) versus 3283 of 4215 at 300bp
(22% excluded), and every excluded one lies past the position where its fragment could still exceed
60bp. At 75bp this effectively restricts the evaluation to start codons in the first ~15bp of a
read. The filter is applied identically to the ground truth and to all three models' predictions, so
the comparison between callers is unaffected - but recall here means "of the start codons in
sufficiently long CDS fragments", and these counts should not be read as how many start codons the
test set contains. `count_start_and_stops_in_test_sets.ipynb` counts the unfiltered labels directly.

Verified on real data that the two sources agree where they overlap: every counted position really
does carry a ground-truth start label (`label == 2`), and no counted position lacks one - so the
denominator is a clean subset of the true start codons, never a superset.

**The border stratum is far from negligible at short read lengths.** `testset_dict.pkl` keeps only
CDS fragments longer than 60bp, and a fragment running from a start codon to the end of the read is
`L - s + 1` long, so a start at read position `s` is only retained when `s < L - 59`. That leaves
just `(L - 60) / 3` in-frame positions per frame at which a start codon can be retained at all, of
which the read's first codon is one - so the border share of all evaluable start codons is roughly
`3 / (L - 60)`, **not** `3 / L`. That is about 20% at 75bp, 7% at 100bp, 3% at 150bp and well under
1% by 700bp, which the `border % of B` column above should track closely. The two assessments
therefore diverge substantially at 75-100bp and barely at all at the long read lengths.

In [95]:
def _counts_row(length, label, excl, incl, border):
    """One row of the ground-truth counts table: the two assessments' counts, the border
    stratum that separates them, and the border's share of B."""
    return {
        'read_length': length,
        'codon_type': label,
        'A: border-excluded': excl,
        'B: border-inclusive': incl,
        'border-only stratum': border,
        'border % of B': 100 * border / incl if incl else np.nan,
    }


counts_rows = []
for length in read_lengths:
    for ct in codon_types:
        counts_rows.append(_counts_row(
            length, codon_type_labels[ct],
            actual_start_counts_excl[length][ct],
            actual_start_counts_incl[length][ct],
            border_start_counts[length][ct]))
    # Total for this read length's testset. ATG and non-ATG partition the ground truth
    # (split_by_codon_type sends every start whose codon is not ATG to non_ATG), so summing the
    # two codon types gives the complete annotated start codon occurrence count for the testset.
    counts_rows.append(_counts_row(
        length, 'Total',
        sum(actual_start_counts_excl[length].values()),
        sum(actual_start_counts_incl[length].values()),
        sum(border_start_counts[length].values())))
counts_df = pd.DataFrame(counts_rows)
print("True start codon occurrences behind each assessment:")
display(counts_df.round(2))

print("\nTotal annotated start codon occurrences per testset (ATG + non-ATG):")
for length in read_lengths:
    n_excl = sum(actual_start_counts_excl[length].values())
    n_incl = sum(actual_start_counts_incl[length].values())
    print(f"  {length}bp: A (border-excluded) = {n_excl:,}   B (border-inclusive) = {n_incl:,}")

# Consistency: B's ground truth is exactly A's plus the border stratum, by construction.
for length in read_lengths:
    for ct in codon_types:
        assert (actual_start_counts_excl[length][ct] + border_start_counts[length][ct]
                == actual_start_counts_incl[length][ct]), f"border stratum mismatch at {length}bp {ct}"
print("Border stratum accounts exactly for the difference between the two ground truths.")

print("\nWhich non-ATG codons occur (should be dominated by GTG and TTG, the standard alternative "
      "starts under translation table 11):")
for length in read_lengths:
    total = sum(non_atg_codon_identity_counts[length].values())
    print(f"  {length}bp (n={total}): {dict(non_atg_codon_identity_counts[length].most_common())}")


True start codon occurrences behind each assessment:


,read_length,codon_type,A: border-excluded,B: border-inclusive,border-only stratum,border % of B
0,75,Canonical (ATG),66061,86033,19972,23.21
1,75,Non-canonical (non-ATG),14386,18705,4319,23.09
2,75,Total,80447,104738,24291,23.19
3,100,Canonical (ATG),172121,187044,14923,7.98
4,100,Non-canonical (non-ATG),37639,40824,3185,7.80
5,100,Total,209760,227868,18108,7.95
6,150,Canonical (ATG),275459,285304,9845,3.45
7,150,Non-canonical (non-ATG),60055,62157,2102,3.38
8,150,Total,335514,347461,11947,3.44
9,300,Canonical (ATG),371896,376859,4963,1.32



Total annotated start codon occurrences per testset (ATG + non-ATG):
  75bp: A (border-excluded) = 80,447   B (border-inclusive) = 104,738
  100bp: A (border-excluded) = 209,760   B (border-inclusive) = 227,868
  150bp: A (border-excluded) = 335,514   B (border-inclusive) = 347,461
  300bp: A (border-excluded) = 452,243   B (border-inclusive) = 458,327
  700bp: A (border-excluded) = 507,081   B (border-inclusive) = 509,688
  1000bp: A (border-excluded) = 512,201   B (border-inclusive) = 513,968
Border stratum accounts exactly for the difference between the two ground truths.

Which non-ATG codons occur (should be dominated by GTG and TTG, the standard alternative starts under translation table 11):
  75bp (n=18705): {'GTG': 13332, 'TTG': 4385, 'CTG': 280, 'ATT': 270, 'ATC': 223, 'ATA': 214, 'AAT': 1}
  100bp (n=40824): {'GTG': 29077, 'TTG': 9634, 'CTG': 603, 'ATT': 547, 'ATC': 497, 'ATA': 462, 'AAA': 2, 'GAC': 2}
  150bp (n=62157): {'GTG': 44289, 'TTG': 14658, 'CTG': 928, 'ATT': 822, 

## Pooled (unweighted) scores and the genome-level cluster bootstrap

TP/FP/FN are summed across all genomes first and the metric computed once from those totals, so
each start codon occurrence carries equal weight and no genome is normalised or weighted.

`bootstrap_pooled_std` is the genome-level cluster bootstrap from
`cds_level_metagenome_model_comparison.ipynb`, unchanged: resample whole genomes with replacement,
sum their raw TP/FP/FN into one pooled count per replicate, recompute the pooled score, and report
the SD across replicates as a standard error (not a percentile interval). Reads within a genome are
correlated, so genomes are the resampling unit. `N_BOOTSTRAP = 1000` and `np.random.default_rng(42)`
match that notebook. A combination needs at least 2 contributing genomes; below that the SE is NaN
and only a bare point estimate is shown.

In [96]:
def safe_div(a, b):
    return a / b if b > 0 else 0.0


N_BOOTSTRAP = 1000
_bootstrap_rng = np.random.default_rng(42)


def pooled_scores(tp, fp, fn):
    '''Vectorized pooled precision/recall/F1 - tp/fp/fn may be scalars or arrays (one bootstrap
    replicate per element). Mirrors pooled_scores in cds_level_metagenome_model_comparison.ipynb.'''
    tp = np.asarray(tp, dtype=float)
    fp = np.asarray(fp, dtype=float)
    fn = np.asarray(fn, dtype=float)
    p = np.divide(tp, tp + fp, out=np.zeros_like(tp), where=(tp + fp) > 0)
    r = np.divide(tp, tp + fn, out=np.zeros_like(tp), where=(tp + fn) > 0)
    f1 = np.divide(2 * p * r, p + r, out=np.zeros_like(tp), where=(p + r) > 0)
    return {'precision': p, 'recall': r, 'f1': f1}


def bootstrap_pooled_std(genome_counts_dict, n_boot=N_BOOTSTRAP, rng=_bootstrap_rng):
    '''Genome-level cluster bootstrap standard error for pooled precision/recall/F1. Resamples
    whole genomes with replacement n_boot times, sums each resample's raw TP/FP/FN into one pooled
    count per replicate (exactly how the point estimate pools all reads), and returns the SD across
    replicates. NaN if fewer than 2 genomes contributed - mirrors bootstrap_pooled_std in
    cds_level_metagenome_model_comparison.ipynb.'''
    genome_ids = list(genome_counts_dict.keys())
    n = len(genome_ids)
    if n < 2:
        return {m: np.nan for m in metric_keys}

    tp_arr = np.array([genome_counts_dict[g]['tp'] for g in genome_ids], dtype=float)
    fp_arr = np.array([genome_counts_dict[g]['fp'] for g in genome_ids], dtype=float)
    fn_arr = np.array([genome_counts_dict[g]['fn'] for g in genome_ids], dtype=float)

    idx = rng.integers(0, n, size=(n_boot, n))  # resample genome indices with replacement
    boot = pooled_scores(tp_arr[idx].sum(axis=1), fp_arr[idx].sum(axis=1), fn_arr[idx].sum(axis=1))
    return {m: float(values.std(ddof=1)) for m, values in boot.items()}


def compute_metrics_and_se(results, per_genome_counts, has_data, models):
    '''Pooled point estimates + genome-level bootstrap SE for one assessment.
    Returns (metrics, metrics_std, n_genomes), each keyed [model][f'{codon_type}_{metric}'] with a
    list indexed by read_lengths.'''
    metrics = {m: {f'{ct}_{k}': [] for ct in codon_types for k in metric_keys} for m in models}
    metrics_std = {m: {f'{ct}_{k}': [] for ct in codon_types for k in metric_keys} for m in models}
    n_genomes = {length: {m: 0 for m in models} for length in read_lengths}

    for length in read_lengths:
        for model in models:
            if not has_data[length][model]:
                for key in metrics[model]:
                    metrics[model][key].append(np.nan)
                    metrics_std[model][key].append(np.nan)
                continue
            for ct in codon_types:
                r = results[length][model][ct]
                prec = safe_div(r['tp'], r['tp'] + r['fp'])
                rec = safe_div(r['tp'], r['tp'] + r['fn'])
                metrics[model][f'{ct}_precision'].append(prec)
                metrics[model][f'{ct}_recall'].append(rec)
                metrics[model][f'{ct}_f1'].append(safe_div(2 * prec * rec, prec + rec))

                boot = bootstrap_pooled_std(per_genome_counts[length][model][ct])
                for k in metric_keys:
                    metrics_std[model][f'{ct}_{k}'].append(boot[k])
            n_genomes[length][model] = len(per_genome_counts[length][model]['ATG'])
    return metrics, metrics_std, n_genomes


metrics_excl, metrics_std_excl, n_genomes_excl = compute_metrics_and_se(
    results_excl, per_genome_counts_excl, has_data_excl, model_names)
metrics_incl, metrics_std_incl, n_genomes_incl = compute_metrics_and_se(
    results_incl, per_genome_counts_incl, has_data_incl, border_inclusive_models)

# Self-check: per_genome_counts and results are independent accumulations of the same numbers.
for tag, res, pgc, hd, models in (("A", results_excl, per_genome_counts_excl, has_data_excl, model_names),
                                  ("B", results_incl, per_genome_counts_incl, has_data_incl, border_inclusive_models)):
    for length in read_lengths:
        for model in models:
            if not hd[length][model]:
                continue
            for ct in codon_types:
                for k in ('tp', 'fp', 'fn'):
                    assert res[length][model][ct][k] == sum(c[k] for c in pgc[length][model][ct].values()), \
                        f"assessment {tag}: pooled/per-genome mismatch at {length}bp {model} {ct} {k}"

# Self-check: for each model, TP + FN must equal the ground-truth occurrence count tallied in
# the separate pass above. It holds because a prediction landing exactly on a true start is
# classified by the read's own codon there, so it always lands in the same codon-type bucket as
# the truth it matched - which makes TP + FN the full count of true starts of that type. Any
# divergence would mean the scoring loop and the ground-truth loop disagree about what counts as
# a start, which is exactly the class of bug this notebook is most exposed to.
for tag, res, hd, models, truth_counts in (
        ("A", results_excl, has_data_excl, model_names, actual_start_counts_excl),
        ("B", results_incl, has_data_incl, border_inclusive_models, actual_start_counts_incl)):
    for length in read_lengths:
        for model in models:
            if not hd[length][model]:
                continue
            for ct in codon_types:
                r = res[length][model][ct]
                assert r['tp'] + r['fn'] == truth_counts[length][ct], (
                    f"assessment {tag}: {model} at {length}bp {ct}: tp+fn={r['tp'] + r['fn']} "
                    f"but ground truth counted {truth_counts[length][ct]}")

print(f"Bootstrap: {N_BOOTSTRAP} genome-level resamples, seed 42.")
print("Per-genome counts sum to the pooled counts for both assessments.")
print("TP + FN matches the independently counted ground truth for every model and codon type.")

Bootstrap: 1000 genome-level resamples, seed 42.
Per-genome counts sum to the pooled counts for both assessments.
TP + FN matches the independently counted ground truth for every model and codon type.


In [97]:
def fmt_pm(v, se, pm=' ± '):
    '''"0.842 ± 0.006", or a bare point estimate where the bootstrap SE is undefined (<2 genomes).'''
    if np.isnan(v):
        return '--'
    return f'{v:.3f}' if np.isnan(se) else f'{v:.3f}{pm}{se:.3f}'


def assessment_frame(metrics, metrics_std, has_data, n_genomes, models, label):
    return pd.DataFrame([
        {
            'assessment': label,
            'read_length': length,
            'model': model_display_names[model],
            'codon_type': codon_type_labels[ct],
            'n_genomes': n_genomes[length][model],
            **{metric_display[k]: fmt_pm(metrics[model][f'{ct}_{k}'][i], metrics_std[model][f'{ct}_{k}'][i])
               for k in ('f1', 'precision', 'recall')},
        }
        for i, length in enumerate(read_lengths)
        for model in models
        for ct in codon_types
        if has_data[length][model]
    ])


print("A: border-excluded (all three models)")
display(assessment_frame(metrics_excl, metrics_std_excl, has_data_excl, n_genomes_excl,
                         model_names, 'A: border-excluded'))
print("\nB: border-inclusive (models with an explicit start-codon call)")
display(assessment_frame(metrics_incl, metrics_std_incl, has_data_incl, n_genomes_incl,
                         border_inclusive_models, 'B: border-inclusive'))

A: border-excluded (all three models)


,assessment,read_length,model,codon_type,n_genomes,F1,Precision,Sensitivity
0,A: border-excluded,75,FGS (Complete),Canonical (ATG),212,0.084 ± 0.002,0.174 ± 0.004,0.056 ± 0.002
1,A: border-excluded,75,FGS (Complete),Non-canonical (non-ATG),212,0.029 ± 0.002,0.041 ± 0.003,0.022 ± 0.002
2,A: border-excluded,75,MetaProdigal,Canonical (ATG),212,0.178 ± 0.003,0.415 ± 0.008,0.113 ± 0.002
3,A: border-excluded,75,MetaProdigal,Non-canonical (non-ATG),212,0.038 ± 0.002,0.165 ± 0.010,0.021 ± 0.001
4,A: border-excluded,75,DeepCDS N,Canonical (ATG),212,0.504 ± 0.007,0.762 ± 0.006,0.377 ± 0.006
5,A: border-excluded,75,DeepCDS N,Non-canonical (non-ATG),212,0.154 ± 0.006,0.740 ± 0.013,0.086 ± 0.004
6,A: border-excluded,100,FGS (Complete),Canonical (ATG),212,0.310 ± 0.003,0.436 ± 0.007,0.241 ± 0.003
7,A: border-excluded,100,FGS (Complete),Non-canonical (non-ATG),212,0.128 ± 0.004,0.180 ± 0.007,0.100 ± 0.003
8,A: border-excluded,100,MetaProdigal,Canonical (ATG),212,0.493 ± 0.005,0.536 ± 0.008,0.456 ± 0.005
9,A: border-excluded,100,MetaProdigal,Non-canonical (non-ATG),212,0.217 ± 0.005,0.283 ± 0.008,0.175 ± 0.005



B: border-inclusive (models with an explicit start-codon call)


,assessment,read_length,model,codon_type,n_genomes,F1,Precision,Sensitivity
0,B: border-inclusive,75,MetaProdigal,Canonical (ATG),212,0.143 ± 0.002,0.415 ± 0.008,0.087 ± 0.001
1,B: border-inclusive,75,MetaProdigal,Non-canonical (non-ATG),212,0.030 ± 0.002,0.165 ± 0.010,0.016 ± 0.001
2,B: border-inclusive,75,DeepCDS N,Canonical (ATG),212,0.471 ± 0.006,0.762 ± 0.006,0.340 ± 0.006
3,B: border-inclusive,75,DeepCDS N,Non-canonical (non-ATG),212,0.131 ± 0.005,0.731 ± 0.012,0.072 ± 0.003
4,B: border-inclusive,100,MetaProdigal,Canonical (ATG),212,0.471 ± 0.005,0.536 ± 0.008,0.420 ± 0.005
5,B: border-inclusive,100,MetaProdigal,Non-canonical (non-ATG),212,0.206 ± 0.005,0.283 ± 0.008,0.162 ± 0.004
6,B: border-inclusive,100,DeepCDS N,Canonical (ATG),212,0.710 ± 0.005,0.836 ± 0.004,0.617 ± 0.005
7,B: border-inclusive,100,DeepCDS N,Non-canonical (non-ATG),212,0.426 ± 0.007,0.784 ± 0.005,0.293 ± 0.007
8,B: border-inclusive,150,MetaProdigal,Canonical (ATG),212,0.633 ± 0.004,0.649 ± 0.007,0.618 ± 0.004
9,B: border-inclusive,150,MetaProdigal,Non-canonical (non-ATG),212,0.356 ± 0.006,0.424 ± 0.009,0.307 ± 0.007


### Diagnostic: the border stratum on its own

The positions assessment A drops and B keeps, scored in isolation (both prediction and truth
restricted to the read's first in-frame codon). This is where the two models' handling of read
edges is directly visible, instead of being diluted into the combined figure: Prodigal marks a gene
whose 5' end reaches the sequence boundary as `partial=1x` with `start_type=Edge` and claims no
start codon there, so its true positives on this stratum are expected to be zero or near it.

`claim rate` is how often the model claimed *any* start at a first-codon position (TP + FP)
relative to the number of true border starts - it separates "declined to call" from "called the
wrong thing".

In [98]:
border_rows = []
for length in read_lengths:
    for model in border_inclusive_models:
        if not has_data_incl[length][model]:
            continue
        for ct in codon_types:
            c = border_only_counts[length][model][ct]
            tp, fp, fn = c['tp'], c['fp'], c['fn']
            n_true = tp + fn
            border_rows.append({
                'read_length': length,
                'model': model_display_names[model],
                'codon_type': codon_type_labels[ct],
                'true border starts': n_true,
                'tp': tp, 'fp': fp, 'fn': fn,
                'recall': safe_div(tp, n_true),
                'precision': safe_div(tp, tp + fp),
                'claim rate': safe_div(tp + fp, n_true),
            })
border_df = pd.DataFrame(border_rows)
display(border_df.round(4))

for model in border_inclusive_models:
    sub = border_df[border_df['model'] == model_display_names[model]]
    if sub.empty:
        continue
    print(f"{model_display_names[model]}: {int(sub['tp'].sum())} TP / {int(sub['true border starts'].sum())} "
          f"true border starts across all read lengths "
          f"(claimed a start at a first-codon position {int(sub['tp'].sum() + sub['fp'].sum())} times)")

,read_length,model,codon_type,true border starts,tp,fp,fn,recall,precision,claim rate
0,75,MetaProdigal,Canonical (ATG),19972,0,0,19972,0.0000,0.0000,0.0000
1,75,MetaProdigal,Non-canonical (non-ATG),4319,0,0,4319,0.0000,0.0000,0.0000
2,75,DeepCDS N,Canonical (ATG),19972,4414,1373,15558,0.2210,0.7627,0.2898
3,75,DeepCDS N,Non-canonical (non-ATG),4319,112,61,4207,0.0259,0.6474,0.0401
4,100,MetaProdigal,Canonical (ATG),14923,0,0,14923,0.0000,0.0000,0.0000
5,100,MetaProdigal,Non-canonical (non-ATG),3185,0,0,3185,0.0000,0.0000,0.0000
6,100,DeepCDS N,Canonical (ATG),14923,4848,1512,10075,0.3249,0.7623,0.4262
7,100,DeepCDS N,Non-canonical (non-ATG),3185,178,93,3007,0.0559,0.6568,0.0851
8,150,MetaProdigal,Canonical (ATG),9845,0,0,9845,0.0000,0.0000,0.0000
9,150,MetaProdigal,Non-canonical (non-ATG),2102,0,0,2102,0.0000,0.0000,0.0000


MetaProdigal: 0 TP / 64804 true border starts across all read lengths (claimed a start at a first-codon position 0 times)
DeepCDS N: 20013 TP / 64804 true border starts across all read lengths (claimed a start at a first-codon position 26495 times)


## LaTeX tables

One table per assessment: ground-truth ATG/non-ATG occurrence counts, then the pooled score for
each codon type as point estimate $\pm$ genome-level cluster bootstrap standard error.

In [99]:
_METRIC_TABLE_LABELS = {'f1': 'F1 score', 'precision': 'Precision', 'recall': 'Sensitivity'}


def fmt_point_se(v, se, bold=False):
    """'0.842 $\\pm$ 0.006', or just '0.842' if the bootstrap SE is undefined (<2 genomes).
    Only the point estimate is emboldened, never the standard error - the SE is not the quantity
    being compared, and bolding it too makes the column hard to scan."""
    if np.isnan(v):
        return '--'
    point = rf'\textbf{{{v:.3f}}}' if bold else f'{v:.3f}'
    return point if np.isnan(se) else f'{point} $\\pm$ {se:.3f}'


def build_latex_table(metrics, metrics_std, has_data, actual_counts, models, label,
                      caption=None):
    """Grouped by codon type first, then metric, then model - so the models sit on consecutive
    rows and read as a direct comparison, and everything about the canonical case is finished
    before the non-canonical case begins. Each codon type's ground-truth occurrence count heads
    its own block, next to the scores that use it as a denominator.

    Within each (codon type, metric) block the best model per read length is emboldened; ties are
    all emboldened and a model with no data is '--' and never counts as best. The caption's closing
    sentence about what bold implies is DERIVED from the numbers rather than asserted: the
    separation between the best and runner-up is measured in combined standard errors, so the
    statement stays true if the data changes."""
    col_fmt = 'l' + 'c' * len(read_lengths)
    n_cols = 1 + len(read_lengths)
    separations, any_overlap = [], False

    body = []
    for ct in codon_types:
        body.append(r'\midrule')
        body.append(rf'\multicolumn{{{n_cols}}}{{l}}{{\textbf{{{codon_type_labels[ct]} start codons}}}} \\')
        body.append(r'\midrule')
        counts_row = ' & '.join(f'{actual_counts[length][ct]:,}'.replace(',', r'\,') for length in read_lengths)
        body.append(rf'\makecell[l]{{True start codon\\occurrences}} & {counts_row} \\')

        for m in ('f1', 'precision', 'recall'):
            best_per_length = []
            for i, length in enumerate(read_lengths):
                col = [(metrics[mo][f'{ct}_{m}'][i], metrics_std[mo][f'{ct}_{m}'][i])
                       for mo in models if has_data[length][mo]]
                col = [(v, s) for v, s in col if not np.isnan(v)]
                if not col:
                    best_per_length.append(None)
                    continue
                col.sort(key=lambda vs: -vs[0])
                best_per_length.append(col[0][0])
                if len(col) > 1:
                    (v1, s1), (v2, s2) = col[0], col[1]
                    s1 = 0.0 if np.isnan(s1) else s1
                    s2 = 0.0 if np.isnan(s2) else s2
                    combined = float(np.hypot(s1, s2))
                    if combined > 0:
                        separations.append((v1 - v2) / combined)
                    if v1 - s1 <= v2 + s2:
                        any_overlap = True

            body.append(r'\addlinespace')
            body.append(rf'\multicolumn{{{n_cols}}}{{l}}{{\textbf{{{_METRIC_TABLE_LABELS[m]}}}}} \\')
            for model in models:
                row_cells = []
                for i, length in enumerate(read_lengths):
                    if not has_data[length][model]:
                        row_cells.append('--')
                        continue
                    v = metrics[model][f'{ct}_{m}'][i]
                    se = metrics_std[model][f'{ct}_{m}'][i]
                    is_best = (best_per_length[i] is not None and not np.isnan(v)
                               and v == best_per_length[i])
                    row_cells.append(fmt_point_se(v, se, bold=is_best))
                body.append(rf'\quad {model_display_names[model]} & ' + ' & '.join(row_cells) + r' \\')

    if not separations:
        note = '% bold marks the best value per read length.'
    elif any_overlap:
        note = ('% bold marks the best value per read length. NOTE: in at least one column the '
                'best and runner-up +/-1SE intervals OVERLAP, so bold alone does not establish '
                'a distinguishable difference there.')
    else:
        note = (f'% bold marks the best value per read length; the best model beats the '
                f'runner-up by at least {min(separations):.1f} combined standard errors in '
                f'every column, so no bolded entry rests on an indistinguishable margin.')

    lines = [r'\begin{table}[ht]', r'\centering', r'\footnotesize',
             *( [rf'\caption{{{caption}}}'] if caption else [] ),
             rf'\label{{{label}}}',
             rf'\begin{{tabular}}{{{col_fmt}}}', r'\toprule',
             '& ' + ' & '.join(f'{l}bp' for l in read_lengths) + r' \\']
    lines += body
    lines += [r'\bottomrule', r'\end{tabular}', r'\end{table}', note]
    return '\n'.join(lines)


print(build_latex_table(
    metrics_excl, metrics_std_excl, has_data_excl, actual_start_counts_excl, model_names,
    label='tab:atg_vs_non_atg_start_codon_border_excluded'))

print('\n\n')

print(build_latex_table(
    metrics_incl, metrics_std_incl, has_data_incl, actual_start_counts_incl, border_inclusive_models,
    label='tab:atg_vs_non_atg_start_codon_border_inclusive'))


\begin{table}[ht]
\centering
\footnotesize
\label{tab:atg_vs_non_atg_start_codon_border_excluded}
\begin{tabular}{lcccccc}
\toprule
& 75bp & 100bp & 150bp & 300bp & 700bp & 1000bp \\
\midrule
\multicolumn{7}{l}{\textbf{Canonical (ATG) start codons}} \\
\midrule
\makecell[l]{True start codon\\occurrences} & 66\,061 & 172\,121 & 275\,459 & 371\,896 & 417\,683 & 422\,146 \\
\addlinespace
\multicolumn{7}{l}{\textbf{F1 score}} \\
\quad FGS (Complete) & 0.084 $\pm$ 0.002 & 0.310 $\pm$ 0.003 & 0.507 $\pm$ 0.004 & 0.640 $\pm$ 0.003 & 0.679 $\pm$ 0.003 & 0.685 $\pm$ 0.003 \\
\quad MetaProdigal & 0.178 $\pm$ 0.003 & 0.493 $\pm$ 0.005 & 0.644 $\pm$ 0.005 & 0.795 $\pm$ 0.004 & 0.869 $\pm$ 0.003 & 0.882 $\pm$ 0.003 \\
\quad DeepCDS N & \textbf{0.504} $\pm$ 0.007 & \textbf{0.728} $\pm$ 0.005 & \textbf{0.844} $\pm$ 0.004 & \textbf{0.912} $\pm$ 0.003 & \textbf{0.927} $\pm$ 0.002 & \textbf{0.931} $\pm$ 0.002 \\
\addlinespace
\multicolumn{7}{l}{\textbf{Precision}} \\
\quad FGS (Complete) & 0.174 $\pm$ 0